# EXPERIMENT 1 (FILTERING) - Downstream Benchmark

Model A trains on the full training split, Model B on the prompt-filtered split.
Both are scored on identical evaluation sets, including leakage-free variants.

The saved JSON contains one entry per model from the reference table and nothing else:
`raw_train`, `filtered_train`, `published`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, ".")

from src.common.config import SEED
from src.common.data import (
    create_fixed_splits,
    duplicate_summary,
    leakage_free_subset,
    load_bnsentmix,
)
from src.common.metrics import (
    build_comparison_table,
    format_comparison_table,
    save_table,
    sentiment_metrics,
)
from src.filtering.classifiers import CLASSICAL_MODELS
from src.filtering.prompts import decisions_path, load_decisions, prompt_version_label

PROVIDER = "groq"
MODEL_NAME = "qwen/qwen3.8-27b"
PROMPT_VERSION = "v3"
DECISIONS_PATH = decisions_path(PROVIDER, MODEL_NAME, PROMPT_VERSION)

dataset = load_bnsentmix()
splits = create_fixed_splits(dataset)
raw_train_data = splits["train"]
validation_data = splits["validation"]
test_data = splits["test"]

decisions = load_decisions(DECISIONS_PATH)
keep = [index for index in range(len(raw_train_data)) if decisions[index]]
filtered_train_data = raw_train_data.select(keep)

validation_clean, validation_overlap = leakage_free_subset(validation_data, raw_train_data)
test_clean, test_overlap = leakage_free_subset(test_data, raw_train_data)

print("Decisions file:", DECISIONS_PATH.name)
print("Raw training rows:", len(raw_train_data))
print("Filtered training rows:", len(filtered_train_data))
print("Duplicate audit:", duplicate_summary(dataset))
print(f"Validation: {len(validation_data)} -> {len(validation_clean)} "
      f"(removed {validation_overlap['rows_removed']})")
print(f"Test: {len(test_data)} -> {len(test_clean)} (removed {test_overlap['rows_removed']})")


## Harness


In [ ]:
EVAL_SETS = {
    "validation": validation_data,
    "test": test_data,
    "validation_clean": validation_clean,
    "test_clean": test_clean,
}


def run_model(model_name, builder):
    results = {}
    for condition, train_data in (
        ("raw_train", raw_train_data),
        ("filtered_train", filtered_train_data),
    ):
        estimator = builder()
        estimator.fit(train_data["Sentence"], train_data["Label"])
        for eval_name, eval_data in EVAL_SETS.items():
            predictions = estimator.predict(eval_data["Sentence"])
            results[f"{condition}__{eval_name}"] = {
                "n_train": len(train_data),
                "n_eval": len(eval_data),
                **sentiment_metrics(eval_data["Label"], predictions),
            }
    return results


def summarize(results):
    rows = []
    for eval_name in ("validation", "test"):
        for condition in ("raw_train", "filtered_train"):
            block = results[f"{condition}__{eval_name}"]["weighted"]
            rows.append((
                eval_name,
                condition,
                block["accuracy"],
                block["precision"],
                block["recall"],
                block["f1"],
            ))
    return rows


## Run and Save


In [ ]:
MODELS_TO_RUN = ["logistic-regression", "random-forest", "svm"]

model_entries = {}

for model_name in MODELS_TO_RUN:
    results = run_model(model_name, CLASSICAL_MODELS[model_name])
    model_entries[model_name] = {"conditions": results}

    print(f"=== {model_name} ===")
    print(f"{'eval set':<12}{'condition':<16}{'acc':>8}{'prec':>8}{'rec':>8}{'f1':>8}")
    for eval_name, condition, acc, precision, recall, f1 in summarize(results):
        print(f"{eval_name:<12}{condition:<16}{acc:>8.4f}{precision:>8.4f}{recall:>8.4f}{f1:>8.4f}")
    print()

table = build_comparison_table(model_entries)
TABLE_PATH = save_table(
    table,
    "filtering",
    provider=PROVIDER,
    model=MODEL_NAME,
    prompt=prompt_version_label(PROMPT_VERSION),
    seed=SEED,
)

print(format_comparison_table(table))
print("Saved:", TABLE_PATH.name)
